# 🌤️ Weather Forecasting MLOps — Model Training
**Train Prophet + LSTM on Google Colab (GPU)**

- **Prophet**: 6 cities × (hourly + daily) = 12 models
- **LSTM**: 3 cities (HN, ĐN, HCM) × (hourly + daily) = 2 multi-city models

LSTM cities: Hà Nội (Bắc), Đà Nẵng (Trung), HCM (Nam)

## 1️⃣ Setup

In [ ]:
!git clone https://github.com/LinhThaoPham/weather-forecasting-mlop.git
%cd weather-forecasting-mlop
!pip install -q prophet tensorflow keras pandas numpy scikit-learn statsmodels joblib requests python-dotenv

In [ ]:
import sys, os, json
sys.path.insert(0, os.getcwd())
import tensorflow as tf
print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

## 2️⃣ Fetch Data → SQLite (6 cities, 2 năm)

In [ ]:
from src.data_pipeline.store_data import seed_all_cities
seed_all_cities(days=730, delay=0.5)

In [ ]:
from src.config.db import get_connection
with get_connection() as conn:
    for t in ['weather_historical','weather_forecast','weather_current']:
        c = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
        print(f'{t}: {c:,} rows')

## 3️⃣ Train Prophet (6 cities × hourly + daily)

In [ ]:
import pandas as pd
from src.config.cities import CITIES
from src.config.constants import TRAIN_SPLIT_RATIO, model_filename
from src.data_pipeline.feature_engineering import add_features
from src.models_logic.prophet_model import train_prophet, save_prophet
from src.training.evaluate import evaluate_prophet

TRAIN_DIR = 'models/training_temp'
os.makedirs(TRAIN_DIR, exist_ok=True)
prophet_metrics = {}

def fetch_city_data(city_id):
    with get_connection() as conn:
        rows = conn.execute(
            'SELECT timestamp, temperature, humidity, cloud_cover '
            'FROM weather_historical WHERE city_id=? ORDER BY timestamp',
            (city_id,)).fetchall()
    df = pd.DataFrame(rows, columns=['ds','y','humidity','cloud_cover'])
    df['ds'] = pd.to_datetime(df['ds'])
    return df.dropna(subset=['y'])

for city_id in sorted(CITIES.keys()):
    print(f'\n=== Prophet: {city_id} ===')
    data = fetch_city_data(city_id)
    print(f'  Data: {len(data):,} rows')
    for mode, is_h in [('hourly',True),('daily',False)]:
        df = add_features(data.copy(), is_hourly=is_h)
        s = int(len(df) * TRAIN_SPLIT_RATIO)
        m = train_prophet(df.iloc[:s], is_hourly=is_h)
        save_prophet(m, os.path.join(TRAIN_DIR, model_filename('prophet',mode,city_id)))
        ev = evaluate_prophet(m, df.iloc[s:], mode)
        prophet_metrics[f'prophet_{mode}_{city_id}'] = ev
        print(f'  {mode} MAE: {ev["mae"]}°C')

print('\n✅ Prophet done!')

## 4️⃣ Train LSTM Hourly Multi-City (3 cities × ~17K hours)

In [ ]:
import numpy as np, joblib
from src.config.constants import *
from src.data_pipeline.feature_engineering import (
    build_multi_city_hourly, build_multi_city_daily,
    normalize_multi_city, sliding_window
)
from src.models_logic.lstm_model import create_lstm_model
from src.training.evaluate import evaluate_lstm_multi_city

lstm_metrics = {}

# Hourly
print('=== LSTM Hourly (3 cities: hanoi, danang, hcm) ===')
df_h, h_cities = build_multi_city_hourly()
h_norm, h_scaler = normalize_multi_city(df_h.values)
Xh, yh = sliding_window(h_norm, HOURLY_LOOKBACK, HOURLY_HORIZON)
s = int(len(Xh) * TRAIN_SPLIT_RATIO)
print(f'  Samples: {len(Xh)} | Train: {s} | Val: {len(Xh)-s}')

lstm_h = create_lstm_model(HOURLY_LOOKBACK, HOURLY_HORIZON, NUM_CITIES)
lstm_h.model.summary()
hist_h = lstm_h.train(Xh[:s],yh[:s],Xh[s:],yh[s:], epochs=MAX_EPOCHS, batch_size=BATCH_SIZE)
lstm_h.save(os.path.join(TRAIN_DIR, model_filename('lstm','hourly')))
joblib.dump(h_scaler, os.path.join(TRAIN_DIR, scaler_filename('hourly')))

m = evaluate_lstm_multi_city(lstm_h.model, Xh[s:], yh[s:], h_scaler)
m['val_loss'] = round(float(hist_h.history['val_loss'][-1]),6)
lstm_metrics['lstm_hourly'] = m
print(f'  ✅ MAE: {m["mae"]}°C')

## 5️⃣ Train LSTM Daily Multi-City (3 cities × ~730 days)

In [ ]:
print('=== LSTM Daily (3 cities: hanoi, danang, hcm) ===')
df_d, d_cities = build_multi_city_daily()
d_norm, d_scaler = normalize_multi_city(df_d.values)
Xd, yd = sliding_window(d_norm, DAILY_LOOKBACK, DAILY_HORIZON)
s = int(len(Xd) * TRAIN_SPLIT_RATIO)
print(f'  Samples: {len(Xd)} | Train: {s} | Val: {len(Xd)-s}')

lstm_d = create_lstm_model(DAILY_LOOKBACK, DAILY_HORIZON, NUM_CITIES)
lstm_d.model.summary()
hist_d = lstm_d.train(Xd[:s],yd[:s],Xd[s:],yd[s:], epochs=MAX_EPOCHS, batch_size=BATCH_SIZE)
lstm_d.save(os.path.join(TRAIN_DIR, model_filename('lstm','daily')))
joblib.dump(d_scaler, os.path.join(TRAIN_DIR, scaler_filename('daily')))

m = evaluate_lstm_multi_city(lstm_d.model, Xd[s:], yd[s:], d_scaler)
m['val_loss'] = round(float(hist_d.history['val_loss'][-1]),6)
lstm_metrics['lstm_daily'] = m
print(f'  ✅ MAE: {m["mae"]}°C')

## 6️⃣ Training History

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1,2,figsize=(14,5))
for ax,h,t in [(axes[0],hist_h,'LSTM Hourly'),(axes[1],hist_d,'LSTM Daily')]:
    ax.plot(h.history['loss'],label='Train'); ax.plot(h.history['val_loss'],label='Val')
    ax.set_title(t); ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
    ax.legend(); ax.grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig('training_history.png',dpi=150); plt.show()

## 7️⃣ Deploy & Summary

In [ ]:
import shutil
CURRENT = 'models/current'
os.makedirs(CURRENT, exist_ok=True)
for f in os.listdir(TRAIN_DIR):
    shutil.copy2(os.path.join(TRAIN_DIR,f), os.path.join(CURRENT,f))
    print(f'  ✓ {f}')

all_m = {**prophet_metrics, **lstm_metrics}
from datetime import datetime
reg = {'current_version':f'v_{datetime.now():%Y-%m-%d}','models':[{
    'version':f'v_{datetime.now():%Y-%m-%d}','trained_at':datetime.now().isoformat(),
    'trained_on':'Google Colab (GPU)','metrics':all_m,'status':'active'}]}
with open('models/registry.json','w') as f: json.dump(reg,f,indent=2)

print('\n' + '='*60)
print('📊 RESULTS')
print('='*60)
for n,m in sorted(all_m.items()):
    print(f'  {n:35s} MAE: {m.get("mae","N/A"):>8}°C')
print('='*60)

## 8️⃣ Push to GitHub (hoặc Download)

In [ ]:
GITHUB_TOKEN = ''  # ← Paste token vào đây
if GITHUB_TOKEN:
    !git config user.name 'colab-bot'
    !git config user.email 'colab@train.bot'
    !git remote set-url origin https://{GITHUB_TOKEN}@github.com/LinhThaoPham/weather-forecasting-mlop.git
    !git add models/ data/weather_forecast.db
    !git commit -m "🤖 Colab retrain $(date +'%Y-%m-%d') | 3-city LSTM + 6-city Prophet"
    !git pull --rebase origin main
    !git push origin main
else:
    print('⚠️ Set GITHUB_TOKEN to push, or download below')

In [ ]:
# Download zip thay vì push
!zip -r trained_models.zip models/current/ models/registry.json
from google.colab import files
files.download('trained_models.zip')